In [ ]:
# =========================
# 0) Install first if needed
# %pip install pandas openpyxl xlrd numpy scikit-learn xgboost tensorflow optuna
# =========================

import re
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate, cross_val_score
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

try:
    import xgboost as xgb
    HAS_XGBOOST = True
except ImportError:
    HAS_XGBOOST = False
    print("XGBoost not installed. Skipping XGBoost models.")

try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    HAS_TENSORFLOW = True
except ImportError:
    HAS_TENSORFLOW = False
    print("TensorFlow not installed. Skipping Neural Network models.")

import optuna
from optuna.pruners import MedianPruner

# -------------------------
# 1) File paths - AUTO-DISCOVERED
# -------------------------
import os
DATA_DIR = os.getcwd()

# Map county names to expected file patterns
COUNTIES_MAP = {
    "Nandi": ("NANDI", "Nandi"),
    "Bungoma": ("BUNGOMA", "Bungoma"),
    "Bomet": ("BOMET", "Bomet"),
    "Kakamega": ("KAKAMEGA", "Kakamega"),
    "Nakuru": ("NAKURU", "Nakuru"),
    "Narok": ("NAROK", "Narok"),
    "Trans Nzoia": ("TRANSNZOIA", "Trans Nzoia"),
    "Uasin Gishu": ("UASIN GISHU", "Uasin Gishu"),
    "Elgeyo Marakwet": ("ELGEYO MARAKWET", "Elgeyo Marakwet"),
}

# -------------------------
# 2) Read helpers
# -------------------------
def read_excel_auto(path, header=0):
    if not os.path.exists(path):
        return None
    if str(path).lower().endswith(".xlsx"):
        return pd.read_excel(path, header=header, engine="openpyxl")
    elif str(path).lower().endswith(".xls"):
        return pd.read_excel(path, header=header, engine="xlrd")
    else:
        raise ValueError(f"Unsupported file type: {path}")

def clean_cols(df):
    df = df.copy()
    df.columns = [str(c).strip() for c in df.columns]
    return df

# -------------------------
# 3) Find files in directory
# -------------------------
def find_file_pattern(pattern):
    """Find first file matching pattern (case insensitive)"""
    for fname in os.listdir(DATA_DIR):
        if pattern.lower() in fname.lower():
            return os.path.join(DATA_DIR, fname)
    return None

# -------------------------
# 4) Temperature loader
# -------------------------
def load_temperature_file(path, county_name):
    """Load temperature data (max/min) for March-June"""
    if path is None or not os.path.exists(path):
        return None
    
    try:
        temp = read_excel_auto(path, header=1)
        temp = clean_cols(temp)
        
        date_col, max_col, min_col = None, None, None
        for c in temp.columns:
            cl = str(c).strip().lower()
            if cl == "date":
                date_col = c
            if "max" in cl and "temp" in cl:
                max_col = c
            if "min" in cl and "temp" in cl:
                min_col = c
        
        if date_col is None or max_col is None or min_col is None:
            return None
        
        temp = temp[[date_col, max_col, min_col]].copy()
        temp.columns = ["date", "temp_max_c", "temp_min_c"]
        
        temp["date"] = pd.to_datetime(temp["date"], errors="coerce")
        temp["temp_max_c"] = pd.to_numeric(temp["temp_max_c"], errors="coerce")
        temp["temp_min_c"] = pd.to_numeric(temp["temp_min_c"], errors="coerce")
        temp = temp.dropna(subset=["date"]).copy()
        
        temp["county"] = county_name
        temp["year"] = temp["date"].dt.year
        temp["month"] = temp["date"].dt.month
        
        # Keep March-June only
        temp = temp[temp["month"].between(3, 6)].copy()
        
        # Aggregate to monthly stats
        monthly = temp.groupby(["county", "year", "month"], as_index=False).agg({
            "temp_max_c": "mean",
            "temp_min_c": "mean"
        })
        
        monthly_wide = monthly.pivot_table(
            index=["county", "year"],
            columns="month",
            values=["temp_max_c", "temp_min_c"],
            aggfunc="mean"
        ).reset_index()
        
        monthly_wide.columns = [f"{col[0]}_{col[1]}" if col[1] else col[0] 
                                 for col in monthly_wide.columns.values]
        
        # Simplify to seasonal stats
        result = monthly_wide[["county", "year"]].copy()
        result["temp_max_mean"] = monthly_wide[[c for c in monthly_wide.columns if "temp_max" in c]].mean(axis=1)
        result["temp_min_mean"] = monthly_wide[[c for c in monthly_wide.columns if "temp_min" in c]].mean(axis=1)
        
        return result
    except Exception as e:
        print(f"Error loading temperature file {path}: {e}")
        return None

# -------------------------
# 5) Soil moisture loader
# -------------------------
def load_soil_moisture_file(path, county_name):
    """Load soil moisture data for March-June"""
    if path is None or not os.path.exists(path):
        return None
    
    try:
        soil = read_excel_auto(path, header=0)
        soil = clean_cols(soil)
        
        date_col, moisture_col = None, None
        for c in soil.columns:
            cl = str(c).strip().lower()
            if cl == "date" or "date" in cl:
                date_col = c
            if "soil" in cl and "moisture" in cl:
                moisture_col = c
        
        if date_col is None or moisture_col is None:
            return None
        
        soil = soil[[date_col, moisture_col]].copy()
        soil.columns = ["date", "soil_moisture"]
        
        soil["date"] = pd.to_datetime(soil["date"], errors="coerce")
        soil["soil_moisture"] = pd.to_numeric(soil["soil_moisture"], errors="coerce")
        soil = soil.dropna(subset=["date", "soil_moisture"]).copy()
        
        soil["county"] = county_name
        soil["year"] = soil["date"].dt.year
        soil["month"] = soil["date"].dt.month
        
        # Keep March-June only
        soil = soil[soil["month"].between(3, 6)].copy()
        
        # Aggregate to monthly stats
        monthly = soil.groupby(["county", "year", "month"], as_index=False)["soil_moisture"].agg([
            "mean", "min", "max", "std"
        ]).reset_index()
        
        monthly_wide = monthly.pivot_table(
            index=["county", "year"],
            columns="month",
            values=["mean", "min", "max", "std"],
            aggfunc="first"
        ).reset_index()
        
        result = monthly_wide[["county", "year"]].copy()
        result["soil_moisture_mean"] = monthly_wide[[c for c in monthly_wide.columns if "mean" in c]].mean(axis=1)
        result["soil_moisture_min"] = monthly_wide[[c for c in monthly_wide.columns if "min" in c]].mean(axis=1)
        result["soil_moisture_max"] = monthly_wide[[c for c in monthly_wide.columns if "max" in c]].mean(axis=1)
        
        return result
    except Exception as e:
        print(f"Error loading soil moisture file {path}: {e}")
        return None

# -------------------------
# 6) Potential water deficit loader
# -------------------------
def load_pwd_file(path, county_name):
    """Load potential water deficit data for March-June"""
    if path is None or not os.path.exists(path):
        return None
    
    try:
        pwd = read_excel_auto(path, header=0)
        pwd = clean_cols(pwd)
        
        date_col, pwd_col = None, None
        for c in pwd.columns:
            cl = str(c).strip().lower()
            if cl == "date" or "date" in cl:
                date_col = c
            if "deficit" in cl or "pwd" in cl or "water" in cl:
                pwd_col = c
        
        if date_col is None or pwd_col is None:
            return None
        
        pwd = pwd[[date_col, pwd_col]].copy()
        pwd.columns = ["date", "water_deficit"]
        
        pwd["date"] = pd.to_datetime(pwd["date"], errors="coerce")
        pwd["water_deficit"] = pd.to_numeric(pwd["water_deficit"], errors="coerce")
        pwd = pwd.dropna(subset=["date", "water_deficit"]).copy()
        
        pwd["county"] = county_name
        pwd["year"] = pwd["date"].dt.year
        pwd["month"] = pwd["date"].dt.month
        
        # Keep March-June only
        pwd = pwd[pwd["month"].between(3, 6)].copy()
        
        # Aggregate to monthly totals
        monthly = pwd.groupby(["county", "year", "month"], as_index=False)["water_deficit"].sum()
        
        monthly_wide = monthly.pivot_table(
            index=["county", "year"],
            columns="month",
            values="water_deficit",
            aggfunc="sum"
        ).reset_index()
        
        result = monthly_wide[["county", "year"]].copy()
        result["water_deficit_total"] = monthly_wide[[3, 4, 5, 6]].sum(axis=1, skipna=True)
        result["water_deficit_mean"] = monthly_wide[[3, 4, 5, 6]].mean(axis=1, skipna=True)
        
        return result
    except Exception as e:
        print(f"Error loading water deficit file {path}: {e}")
        return None

# -------------------------
# 7) Rainfall loader (enhanced)
# -------------------------
def load_rainfall_file(path, county_name):
    """Load rainfall data for March-June"""
    if path is None or not os.path.exists(path):
        return None
    
    try:
        rain = read_excel_auto(path, header=1)
        rain = clean_cols(rain)
        
        date_col = None
        rain_col = None
        
        for c in rain.columns:
            cl = str(c).strip().lower()
            if cl == "date":
                date_col = c
            if "precipitation" in cl and "mm" in cl:
                rain_col = c
        
        if date_col is None or rain_col is None:
            return None
        
        rain = rain[[date_col, rain_col]].copy()
        rain.columns = ["date", "rainfall_mm"]
        
        rain["date"] = pd.to_datetime(rain["date"], errors="coerce")
        rain["rainfall_mm"] = pd.to_numeric(rain["rainfall_mm"], errors="coerce")
        rain = rain.dropna(subset=["date", "rainfall_mm"]).copy()
        
        rain["county"] = county_name
        rain["year"] = rain["date"].dt.year
        rain["month"] = rain["date"].dt.month
        
        # Keep March-June only
        rain = rain[rain["month"].between(3, 6)].copy()
        
        # Aggregate daily rain to monthly totals
        monthly = (
            rain.groupby(["county", "year", "month"], as_index=False)["rainfall_mm"]
            .sum()
        )
        
        # Pivot month to columns
        monthly_wide = monthly.pivot_table(
            index=["county", "year"],
            columns="month",
            values="rainfall_mm",
            aggfunc="sum"
        ).reset_index()
        
        if 3 in monthly_wide.columns:
            monthly_wide = monthly_wide.rename(columns={
                3: "march_rain",
                4: "april_rain",
                5: "may_rain",
                6: "june_rain"
            })
        
        for col in ["march_rain", "april_rain", "may_rain", "june_rain"]:
            if col not in monthly_wide.columns:
                monthly_wide[col] = 0.0
        
        monthly_wide["total_rain_mar_jun"] = (
            monthly_wide["march_rain"] +
            monthly_wide["april_rain"] +
            monthly_wide["may_rain"] +
            monthly_wide["june_rain"]
        )
        monthly_wide["early_rain"] = monthly_wide["march_rain"] + monthly_wide["april_rain"]
        monthly_wide["late_rain"] = monthly_wide["may_rain"] + monthly_wide["june_rain"]
        monthly_wide["rain_variability"] = monthly_wide[
            ["march_rain", "april_rain", "may_rain", "june_rain"]
        ].std(axis=1)
        
        return monthly_wide
    except Exception as e:
        print(f"Error loading rainfall file {path}: {e}")
        return None

# -------------------------
# 8) Yield loader (enhanced)
# -------------------------
def load_yield_file(path):
    """Load yield data from multiple counties"""
    if not os.path.exists(path):
        raise FileNotFoundError(f"Yield file not found: {path}")
    
    raw = pd.read_excel(path, header=[1, 2], engine="openpyxl")
    
    new_cols = []
    for top, bottom in raw.columns:
        top = "" if pd.isna(top) else str(top).strip()
        bottom = "" if pd.isna(bottom) else str(bottom).strip()
        
        if top.lower() == "county" or bottom.lower() == "county":
            new_cols.append("county")
        elif re.fullmatch(r"\d{4}", top):
            if "yield" in bottom.lower():
                new_cols.append(f"yield_{top}")
            elif "production" in bottom.lower():
                new_cols.append(f"production_{top}")
            elif "harvested" in bottom.lower():
                new_cols.append(f"harvested_area_{top}")
            else:
                new_cols.append(f"{bottom}_{top}")
        else:
            new_cols.append(bottom if bottom else top)
    
    raw.columns = new_cols
    df = raw.copy()
    
    # Keep county + yearly yield columns
    keep_cols = ["county"] + [c for c in df.columns if c.startswith("yield_")]
    yield_df = df[keep_cols].copy()
    
    # Clean county column
    yield_df["county"] = yield_df["county"].astype(str).str.strip()
    yield_df = yield_df[yield_df["county"].notna()]
    yield_df = yield_df[yield_df["county"].str.lower() != "nan"]
    
    # Wide -> long
    long_df = yield_df.melt(
        id_vars="county",
        var_name="year_col",
        value_name="yield_t_ha"
    )
    
    long_df["year"] = long_df["year_col"].str.extract(r"(\d{4})").astype(float)
    long_df["yield_t_ha"] = pd.to_numeric(long_df["yield_t_ha"], errors="coerce")
    long_df = long_df.dropna(subset=["year", "yield_t_ha"]).copy()
    long_df["year"] = long_df["year"].astype(int)
    
    # Standardize county names to match our map
    long_df["county"] = long_df["county"].str.title().str.strip()
    
    # Filter to counties we have data for
    valid_counties = list(COUNTIES_MAP.keys())
    long_df = long_df[long_df["county"].isin(valid_counties)].copy()
    
    print(f"Found yield data for counties: {long_df['county'].unique().tolist()}")
    
    return long_df[["county", "year", "yield_t_ha"]].sort_values(["county", "year"]).reset_index(drop=True)

# -------------------------
# 9) Main enhanced loader
# -------------------------
def load_enhanced_data(yield_file):
    """Load yield + all climate features for all available counties"""
    print("Loading yield data...")
    yield_df = load_yield_file(yield_file)
    
    # Dictionary to store all features by county-year
    all_features = {}
    
    # Load all weather features for all counties
    for county, (file_prefix, county_display) in COUNTIES_MAP.items():
        print(f"\nLoading data for {county}...")
        
        # Get yield data for this county
        county_yield = yield_df[yield_df["county"] == county].copy()
        if len(county_yield) == 0:
            continue
        
        county_data = county_yield[["county", "year", "yield_t_ha"]].copy()
        all_features[county] = county_data
        
        # Rainfall
        rain_path = find_file_pattern(f"{file_prefix} MEAN DAILY RAINFALL")
        if rain_path:
            rain_df = load_rainfall_file(rain_path, county)
            if rain_df is not None:
                rain_cols = [c for c in rain_df.columns if c not in ["county", "year"]]
                county_data = county_data.merge(
                    rain_df[["year"] + rain_cols], 
                    on="year", 
                    how="left"
                )
                print(f"  ✓ Rainfall loaded")
        
        # Temperature
        temp_path = find_file_pattern(f"{file_prefix} DAILY MAX AND MIN TEMPERATURE")
        if temp_path:
            temp_df = load_temperature_file(temp_path, county)
            if temp_df is not None:
                temp_cols = [c for c in temp_df.columns if c not in ["county", "year"]]
                county_data = county_data.merge(
                    temp_df[["year"] + temp_cols], 
                    on="year", 
                    how="left"
                )
                print(f"  ✓ Temperature loaded")
        
        # Soil moisture
        soil_path = find_file_pattern(f"{file_prefix} SOIL MOISTURE")
        if soil_path:
            soil_df = load_soil_moisture_file(soil_path, county)
            if soil_df is not None:
                soil_cols = [c for c in soil_df.columns if c not in ["county", "year"]]
                county_data = county_data.merge(
                    soil_df[["year"] + soil_cols], 
                    on="year", 
                    how="left"
                )
                print(f"  ✓ Soil moisture loaded")
        
        # Water deficit
        pwd_path = find_file_pattern(f"{file_prefix} potential water deficit")
        if pwd_path:
            pwd_df = load_pwd_file(pwd_path, county)
            if pwd_df is not None:
                pwd_cols = [c for c in pwd_df.columns if c not in ["county", "year"]]
                county_data = county_data.merge(
                    pwd_df[["year"] + pwd_cols], 
                    on="year", 
                    how="left"
                )
                print(f"  ✓ Water deficit loaded")
        
        all_features[county] = county_data
    
    # Merge all counties
    print("\nMerging datasets...")
    df = pd.concat(list(all_features.values()), ignore_index=True)
    
    # Feature engineering
    df["prev_yield"] = df.groupby("county")["yield_t_ha"].shift(1)
    
    print(f"\nDataset shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    
    return df

# -------------------------
# 10) Chronological split
# -------------------------
def chronological_split(df, train_end=2017, val_end=2018):
    df = df.copy()
    df["year"] = df["year"].astype(int)
    
    train_df = df[df["year"] <= train_end].copy()
    val_df = df[(df["year"] > train_end) & (df["year"] <= val_end)].copy()
    test_df = df[df["year"] > val_end].copy()
    
    return train_df, val_df, test_df

# -------------------------
# 11) Metrics & evaluation
# -------------------------
def metrics_dict(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred)
    }

# -------------------------
# 12) Prepare data
# -------------------------
print("=" * 60)
print("ENHANCED MAIZE YIELD PREDICTION MODEL")
print("=" * 60)

yield_file = find_file_pattern("Maize-Production") or find_file_pattern("Annual Maize")
if not yield_file:
    raise FileNotFoundError("Could not find yield file")

print(f"Using yield file: {yield_file}")

df = load_enhanced_data(yield_file)

# Fill missing features with forward/backward fill or median
print("\nHandling missing values...")
for col in df.columns:
    if col not in ["county", "year", "yield_t_ha"]:
        # Forward fill by county
        df[col] = df.groupby("county")[col].transform(
            lambda x: x.ffill().bfill()
        )
        # Fill remaining with median
        df[col].fillna(df[col].median(), inplace=True)

train_df, val_df, test_df = chronological_split(df, train_end=2017, val_end=2018)

print(f"Train samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"Test samples: {len(test_df)}")

TARGET = "yield_t_ha"

# Auto-detect feature columns
FEATURES = [c for c in df.columns if c not in ["county", "year", TARGET]]

print(f"\nUsing {len(FEATURES)} features:")
print(f"  {FEATURES}")

X_train = train_df[FEATURES].copy()
y_train = train_df[TARGET].copy()

X_val = val_df[FEATURES].copy()
y_val = val_df[TARGET].copy()

X_test = test_df[FEATURES].copy()
y_test = test_df[TARGET].copy()

# -------------------------
# 13) Preprocessing pipeline
# -------------------------
categorical_features = ["county"] if "county" in FEATURES else []
numeric_features = [c for c in FEATURES if c not in categorical_features]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_features),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
        ]), categorical_features),
    ],
    remainder="drop"
)

# -------------------------
# 14) Model configurations
# -------------------------
models = {
    "DummyMean": DummyRegressor(strategy="mean"),
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "RandomForest": RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42, n_jobs=-1),
    "SVR": SVR(kernel="rbf", C=10.0, epsilon=0.1),
}

if HAS_XGBOOST:
    models["XGBoost"] = xgb.XGBRegressor(
        n_estimators=100,
        max_depth=4,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1
    )

# -------------------------
# 15) Train & evaluate with cross-validation
# -------------------------
print("\n" + "=" * 60)
print("TRAINING MODELS WITH CROSS-VALIDATION")
print("=" * 60)

results = []
pipelines = {}
cv_results = {}

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    print(f"\nTraining {name}...")
    
    pipe = Pipeline([
        ("prep", preprocessor),
        ("model", model)
    ])
    
    # Cross-validation on training set
    cv_scores = cross_validate(
        pipe, X_train, y_train,
        cv=kfold,
        scoring={"mae": "neg_mean_absolute_error", "r2": "r2"},
        n_jobs=-1
    )
    
    cv_mae = -cv_scores["test_mae"].mean()
    cv_mae_std = cv_scores["test_mae"].std()
    cv_r2 = cv_scores["test_r2"].mean()
    
    # Train on full training set
    pipe.fit(X_train, y_train)
    
    # Predict on val and test
    val_pred = pipe.predict(X_val)
    test_pred = pipe.predict(X_test)
    
    val_metrics = metrics_dict(y_val, val_pred)
    test_metrics = metrics_dict(y_test, test_pred)
    
    results.append({
        "model": name,
        "cv_MAE": cv_mae,
        "cv_MAE_std": cv_mae_std,
        "cv_R2": cv_r2,
        "val_MAE": val_metrics["MAE"],
        "val_RMSE": val_metrics["RMSE"],
        "val_R2": val_metrics["R2"],
        "test_MAE": test_metrics["MAE"],
        "test_RMSE": test_metrics["RMSE"],
        "test_R2": test_metrics["R2"],
    })
    
    pipelines[name] = pipe
    cv_results[name] = cv_scores
    
    print(f"  CV MAE: {cv_mae:.4f} (+/- {cv_mae_std:.4f})")
    print(f"  CV R2: {cv_r2:.4f}")
    print(f"  Val RMSE: {val_metrics['RMSE']:.4f}")
    print(f"  Test RMSE: {test_metrics['RMSE']:.4f}")

# -------------------------
# 16) Results summary
# -------------------------
results_df = pd.DataFrame(results).sort_values("test_RMSE").reset_index(drop=True)

print("\n" + "=" * 60)
print("MODEL COMPARISON (sorted by Test RMSE)")
print("=" * 60)
print(results_df.to_string(index=False))

results_df.to_csv("enhanced_model_results.csv", index=False)
print("\n✓ Results saved to enhanced_model_results.csv")

# -------------------------
# 17) Feature importance (for tree-based models)
# -------------------------
print("\n" + "=" * 60)
print("FEATURE IMPORTANCE ANALYSIS")
print("=" * 60)

best_model_name = results_df.iloc[0]["model"]
best_pipe = pipelines[best_model_name]

# Get importances
model_obj = best_pipe.named_steps["model"]

if hasattr(model_obj, 'feature_importances_'):
    try:
        # For tree-based models, use simple feature names
        importances = model_obj.feature_importances_
        feature_names = FEATURES.copy()
        
        if len(importances) == len(feature_names):
            importance_df = pd.DataFrame({
                "feature": feature_names,
                "importance": importances
            }).sort_values("importance", ascending=False)
            
            print(f"\nFeature Importance for {best_model_name}:")
            print(importance_df.to_string(index=False))
            importance_df.to_csv("feature_importance.csv", index=False)
            print("\n✓ Feature importance saved to feature_importance.csv")
    except Exception as e:
        print(f"Could not extract feature importance: {e}")

# -------------------------
# 18) Hyperparameter optimization (XGBoost example)
# -------------------------
if HAS_XGBOOST:
    print("\n" + "=" * 60)
    print("HYPERPARAMETER TUNING WITH OPTUNA")
    print("=" * 60)
    
    def objective(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 50, 300),
            "max_depth": trial.suggest_int("max_depth", 3, 8),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        }
        
        model = xgb.XGBRegressor(**params, random_state=42, n_jobs=-1)
        pipe = Pipeline([
            ("prep", preprocessor),
            ("model", model)
        ])
        
        scores = cross_val_score(pipe, X_train, y_train, cv=3, scoring="r2")
        return scores.mean()
    
    print("\nOptimizing XGBoost hyperparameters (this may take a minute)...")
    sampler = optuna.samplers.TPESampler(seed=42)
    study = optuna.create_study(sampler=sampler, direction="maximize")
    study.optimize(objective, n_trials=20, show_progress_bar=False)
    
    print(f"Best trial CV R2: {study.best_value:.4f}")
    print(f"Best parameters: {study.best_params}")
    
    # Train final model with best params
    best_params = study.best_params
    best_model = xgb.XGBRegressor(**best_params, random_state=42, n_jobs=-1)
    best_pipe = Pipeline([
        ("prep", preprocessor),
        ("model", best_model)
    ])
    best_pipe.fit(X_train, y_train)
    
    test_pred = best_pipe.predict(X_test)
    test_metrics = metrics_dict(y_test, test_pred)
    
    print(f"\nOptimized XGBoost Test RMSE: {test_metrics['RMSE']:.4f}")
    print(f"Optimized XGBoost Test R2: {test_metrics['R2']:.4f}")

# -------------------------
# 19) Predictions on test set
# -------------------------
print("\n" + "=" * 60)
print("SAMPLE PREDICTIONS")
print("=" * 60)

best_pipe = pipelines[best_model_name]
test_pred = best_pipe.predict(X_test)

pred_df = test_df[["county", "year", "yield_t_ha"]].copy()
pred_df["predicted_yield"] = test_pred
pred_df["error"] = pred_df["yield_t_ha"] - pred_df["predicted_yield"]
pred_df["error_pct"] = (pred_df["error"] / pred_df["yield_t_ha"]) * 100

print("\nFirst 10 predictions:")
print(pred_df.head(10)[["county", "year", "yield_t_ha", "predicted_yield", "error_pct"]].to_string(index=False))

pred_df.to_csv("test_predictions.csv", index=False)
print("\n✓ Predictions saved to test_predictions.csv")

print("\n" + "=" * 60)
print("ANALYSIS COMPLETE")
print("=" * 60)
print(f"\nBest performing model: {best_model_name}")
print(f"Test RMSE: {results_df.iloc[0]['test_RMSE']:.4f}")
print(f"Test R2: {results_df.iloc[0]['test_R2']:.4f}")
print(f"\nOutput files generated:")
print("  • enhanced_model_results.csv")
print("  • feature_importance.csv")
print("  • test_predictions.csv")
